In [ ]:
# Convert XML to dict
from sympy import li


def xml_to_dict(element):
    """
    Recursively converts an XML element and its children into a dictionary.
    """
    # Convert attributes and text of the element to a dictionary
    result = {element.tag: {} if element.attrib else None}

    # Add element attributes to the dictionary
    if element.attrib:
        result[element.tag].update((key, value) for key, value in element.attrib.items())

    # Add element text to the dictionary if it exists
    if element.text and element.text.strip():
        text = element.text.strip()
        if result[element.tag]:
            result[element.tag]['text'] = text
        else:
            result[element.tag] = text

    # Convert child elements
    children = list(element)
    if children:
        child_dict = {}
        for child in children:
            child_result = xml_to_dict(child)
            if child.tag in child_dict:
                if isinstance(child_dict[child.tag], list):
                    child_dict[child.tag].append(child_result[child.tag])
                else:
                    child_dict[child.tag] = [child_dict[child.tag], child_result[child.tag]]
            else:
                child_dict.update(child_result)
        if result[element.tag]:
            result[element.tag].update(child_dict)
        else:
            result[element.tag] = child_dict

    return result


def get_deviceset_element(schematic_library, library_name, deviceset_name):
    """
    Extract the entire element of the deviceset from the schematic library.

    Args:
        schematic_library (list): List of libraries from the schematic XML.
        library_name (str): Name of the library containing the deviceset.
        deviceset_name (str): Name of the deviceset to extract.

    Returns:
        dict: The entire element of the deviceset, or None if not found.
    """
    # Find the library with the specified name

    # library = next((lib for lib in schematic_library if lib['name'] == library_name), None)
    library = [lib for lib in schematic_library if lib['name'] == library_name]
    if not library:
        print(f"Library '{library_name}' not found.")
        return None
    

    devicesets = []
    for l in library:
        ds = l.get('devicesets', {}).get('deviceset', [])
        if isinstance(ds, dict):
            ds = [ds]
        elif not isinstance(ds, list):
            ds = []
        devicesets.extend(ds)
    

    deviceset = next((ds for ds in devicesets if ds['name'] == deviceset_name), None)
    if not deviceset:
        print(f"Deviceset '{deviceset_name}' not found in library '{library_name}'.")
        return None

    return deviceset


def extract_deviceset_info(deviceset_xml, device_name):
    """
    Extract deviceset name, symbol, and package from a deviceset XML.

    Args:
        deviceset_xml (dict): The deviceset XML as a dictionary.
        device_name (str): The name of the device to extract information for.

    Returns:
        dict: A dictionary containing the deviceset name, symbol, and package.
    """
    deviceset_info = {
        'name': deviceset_xml.get('name', 'Unknown'),
        'symbols': 'Unknown',
        'packages': ['Unknown']
    }

    # Extract gates and their symbols
    gates = deviceset_xml.get('gates', {}).get('gate', [])
    if isinstance(gates, dict):
        gates = [gates]
    if gates:
        deviceset_info['symbols'] = gates[0].get('symbol', 'Unknown')

    # Extract devices and their packages
    devices = deviceset_xml.get('devices', {}).get('device', [])
    if isinstance(devices, dict):
        devices = [devices]

    # Find the device with the specified device_name
    device_info = next((device for device in devices if device.get('name') == device_name), None)
    if device_info:
        deviceset_info['packages'] = device_info.get('package', 'Unknown')

    return deviceset_info


def map_part_to_deviceset(schematic_parts, schematic_library, schematic_instance):
    """
    Maps parts to their corresponding deviceset names, libraries, and additional attributes,
    including a list of instances with gate-specific information.

    Args:
        schematic_parts (list): List of parts containing deviceset, library, and value information.
        schematic_instance (list): List of instances containing part names and other attributes.

    Returns:
        dict: A dictionary where keys are part names and values are dictionaries containing deviceset, library,
              device, value, symbol, package, and a list of instances with gate-specific information.
    """
    # Create a mapping of part names to deviceset names, libraries, devices, and values
    # part_to_deviceset_and_library = {
    #     part['name']: {
    #         'deviceset': part['deviceset'],
    #         'library': part.get('library', 'Unknown'),
    #         'device': part.get('device', 'Unknown'),  # Include device information
    #         'value': '' if 'PowerSymbols' in part.get('library', 'Unknown') else part.get('value', ''),  # Conditionally include value information
    #         'package': 'Unknown',  # Initialize package information
    #         'instances': []  # Initialize an empty list for instances
    #     }
    #     for part in schematic_parts
    # }
    part_to_deviceset_and_library = {}

    for part in schematic_parts:
        part_name = part['name']
        part_info = {
            'deviceset': part['deviceset'],
            'library': part.get('library', 'Unknown'),
            'device': part.get('device', 'Unknown'),  # Include device information
            # 'value': '' if 'PowerSymbols' in part.get('library', 'Unknown') else part.get('value', ''),  # Conditionally include value information
            'value': part.get('value', part['deviceset']+part.get('device', 'Unknown')),  # Include value information
            'package': 'Unknown',  # Initialize package information
            'instances': [],  # Initialize an empty list for instances
            'value_in_part': part.get('value', 'Unknown')
        }


        # Optional attributes: add only if they exist
        optional_keys = ['library_urn', 'package3d_urn']
        for key in optional_keys:
            if key in part:
                part_info[key] = part[key]

        part_to_deviceset_and_library[part_name] = part_info
        
    # Map instances to their corresponding parts and add gate-specific information
    for instance in schematic_instance:
        
        part_name = instance['part']
        if part_name in part_to_deviceset_and_library:
            library_name = part_to_deviceset_and_library[part_name]['library']
            deviceset_name = part_to_deviceset_and_library[part_name]['deviceset']
            device_name = part_to_deviceset_and_library[part_name]['device']
            deviceset_element = get_deviceset_element(schematic_library, library_name, deviceset_name)

            # Find the symbol for the instance based on the gate name
            gate_name = instance.get('gate', 'Unknown')
            if deviceset_element is None:
                continue
            gates = deviceset_element.get('gates', {}).get('gate', [])
            if isinstance(gates, dict):
                gates = [gates]
            symbol = next((gate.get('symbol', 'Unknown') for gate in gates if gate.get('name') == gate_name), 'Unknown')

            instance_data = {
                'gate': gate_name,
                'x': instance.get('x', 'Unknown'),
                'y': instance.get('y', 'Unknown'),
                'rot': instance.get('rot', 'R0'),
                'symbol': symbol,
                "attribute": instance.get('attribute', [])
            }
            
            part_to_deviceset_and_library[part_name]['instances'].append(instance_data)

            # Update package information if not already set
            if part_to_deviceset_and_library[part_name]['package'] == 'Unknown':
                deviceset_info = extract_deviceset_info(deviceset_element, device_name)
                part_to_deviceset_and_library[part_name]['package'] = deviceset_info['packages']

    # for part in part_to_deviceset_and_library:
    #     print("part: ",part, part_to_deviceset_and_library[part])
        # print("--------------------------------------------------")

    return part_to_deviceset_and_library


def get_symbol_element_of_instance_from_library(library_xml, part_name, instance_gate_name, part_library):
    """
    Extract the symbol element from the library XML based on the part name and instance gate name.

    Args:
        library_xml (list): List of libraries from the schematic XML.
        part_name (str): Name of the part to extract the symbol for.
        instance_gate_name (str): Name of the gate to match the instance.
        part_library (dict): Dictionary containing part names mapped to their library, symbol, and other details.

    Returns:
        dict: The symbol element, or None if not found.
    """
    # Find the library name and symbol using the part name from the part library
    part_info = part_library.get(part_name)
    if not part_info:
        print(f"Part '{part_name}' not found in part library.")
        return None

    library_name = part_info.get('library')
    instances = part_info.get('instances', [])
    if not library_name or not instances:
        print(f"Library or instances not found for part '{part_name}'.")
        return None

    # Find the library with the specified name
    library = [lib for lib in library_xml if lib['name'] == library_name]
    if len(library) == 0:
        print(f"Library '{library_name}' not found.")
        return None

    # Extract symbols for the instance matching the specified gate name
    for instance in instances:
        
        if instance.get('gate') != instance_gate_name:
            continue
        

        symbol_name = instance.get('symbol')
        if not symbol_name:
            print(f"Symbol not found for instance in part '{part_name}'.")
            continue

        # Find the symbol in the library
        #library_symbols = library.get('symbols', {}).get('symbol', [])
        library_symbols = []
        for l in library:
            ds = l.get('symbols', {}).get('symbol', [])
            if isinstance(ds, dict):
                ds = [ds]
            elif not isinstance(ds, list):
                ds = []
            library_symbols.extend(ds)




        if not isinstance(library_symbols, list):
            library_symbols = [library_symbols]

        symbol_element = next((sym for sym in library_symbols if sym['name'] == symbol_name), None)
        if not symbol_element:
            print(2)
            print(f"Symbol '{symbol_name}' not found in library '{library_name}'.")
            continue


        
        
        # print("instance:",instance)
        # print("------------------------")
        # Convert the symbol element into the desired dictionary format
        symbol_data = {
            'name': symbol_element.get('name', 'Unknown'),
            'wire': symbol_element.get('wire', []) if isinstance(symbol_element.get('wire', []), list) else [symbol_element.get('wire', [])],
            'text': instance.get('attribute', []) if isinstance(instance.get('attribute', []), list) else [instance.get('attribute', [])],
            'pin': symbol_element.get('pin', []) if isinstance(symbol_element.get('pin', []), list) else [symbol_element.get('pin', [])],
            'rectangle': symbol_element.get('rectangle', []) if isinstance(symbol_element.get('rectangle', []), list) else [symbol_element.get('rectangle', [])],
            'circle': symbol_element.get('circle', []) if isinstance(symbol_element.get('circle', []), list) else [symbol_element.get('circle', [])],
            'polygon': symbol_element.get('polygon', []) if isinstance(symbol_element.get('polygon', []), list) else [symbol_element.get('polygon', [])],
            'attribute': instance.get('attribute', []),
            'symbol_text': symbol_element.get('text',[]) if isinstance(symbol_element.get('text',[]), list) else [symbol_element.get('text',[])],
            'frame': symbol_element.get('frame', []) if isinstance(symbol_element.get('frame', []), list) else [symbol_element.get('frame', [])],
        }

        # print('symbol_data:', symbol_data)
        # if (symbol_element.get('name', 'Unknown') == "I2C_STANDARD"):
        #     import json
        #     with open("example_symbol_element.json", "w") as f:
        #                 json.dump(symbol_data, f, indent=4)
        # print("symbol_data:",symbol_data)
        # print("--------------------------------------------------")
        return symbol_data
    
    print(f"No matching symbol found for gate '{instance_gate_name}' in part '{part_name}'.")
    return None



import json

def find_connects(data, library_name, deviceset_name, device_name=""):
    """
    Extracts the 'connects' field from the specified device in the deviceset.

    Args:
        data (dict): The full JSON-parsed data structure.
        library_name (str): The name of the library (e.g. "40xx").
        deviceset_name (str): The name of the deviceset (e.g. "4066").
        device_name (str): The name of the specific device (e.g. "", "D", "N").

    Returns:
        list of dicts: The list of 'connect' entries, or None if not found.
    """
    for library in data.get("IC_library", []):
        if library.get("name") == library_name:
            devicesets = library.get("devicesets", {}).get("deviceset", {})
            if isinstance(devicesets, list):
                for ds in devicesets:
                    if ds.get("name") == deviceset_name:
                        # import json
                        # with open("text.json", "w") as f:
                        #     json.dump(ds, f, indent=2)
                        devices = ds.get("devices", {}).get("device")
                        if isinstance(devices, dict):
                            devices = [devices]
                        for device in devices:
                            if device.get("name") == device_name:
                                return device.get("connects", {}).get("connect", [])
            elif devicesets.get("name") == deviceset_name:
                devices = devicesets.get("devices", {}).get("device", [])
                if isinstance(devices, dict):
                    devices = [devices]
                for device in devices:
                    if device.get("name") == device_name:
                        return device.get("connects", {}).get("connect", [])
    return None

  
def retrieve_net_info(schematic_net):
    """
    Extract net information from the schematic net data.

    Args:
        schematic_net (list): List of nets from the schematic XML.

    Returns:
        dict: A dictionary where keys are net names and values are lists of segments,
              with each segment containing lists of pinrefs, wires, labels, and junctions.
    """
    net_info = {}

    # Ensure schematic_net is a list
    if not isinstance(schematic_net, list):
        schematic_net = [schematic_net]

    for net in schematic_net:
        net_name = net.get('name', 'Unknown')
        segments = net.get('segment', [])
        if not isinstance(segments, list):
            segments = [segments]

        # Process each segment
        processed_segments = []
        for segment in segments:
            processed_segment = {
                'pinref': segment.get('pinref', []) if isinstance(segment.get('pinref', []), list) else [segment.get('pinref', [])],
                'wire': segment.get('wire', []) if isinstance(segment.get('wire', []), list) else [segment.get('wire', [])],
                'label': segment.get('label', []) if isinstance(segment.get('label', []), list) else [segment.get('label', [])],
                'junction': segment.get('junction', []) if isinstance(segment.get('junction', []), list) else [segment.get('junction', [])]
            }
            processed_segments.append(processed_segment)

        net_info[net_name] = processed_segments

    return net_info



def process_schematic_file(file_path):
    """
    Process the board file and extract relevant information.

    Args:
        file_path (str): Path to the board file.

    Returns:
        dict: A dictionary containing board information.
    """
    tree = ET.parse(file_path)
    root = tree.getroot()
    xml_dict = xml_to_dict(root)



    check_sheet = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get("sheet", [])
    if isinstance(check_sheet, list):
        raise ValueError(f"Multiple sheets found: {len(check_sheet)}")
    
    
    # Safely retrieve the value of 'busses' from the nested dictionary
    check_busses = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get('sheet', {}).get('busses', {})

    # Print the value of 'busses'
    print("bus:", check_busses)

    # If 'busses' exists, iterate and print its elements
    if check_busses:
        print("Busses found:")
        print(check_busses['name'])
    else:
        print("No busses found.")
    

    # schematic_library = xml_dict['eagle']['drawing']['schematic']['libraries']['library']
    # schematic_parts = xml_dict['eagle']['drawing']['schematic']['parts']["part"]
    # schematic_instance = xml_dict['eagle']['drawing']['schematic']['sheets']["sheet"]['instances']['instance']
    # schematic_net = xml_dict['eagle']['drawing']['schematic']['sheets']["sheet"]['nets']['net']

        # Handle cases where instances, parts, or nets might not exist
    schematic_library = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('libraries', {})
    if schematic_library is None:
        schematic_library = []
    else:
        schematic_library = schematic_library.get("library", []) or []
        if isinstance(schematic_library, dict):
            schematic_library = [schematic_library]

    schematic_parts = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('parts', {})
    if schematic_parts is None:
        schematic_parts = []
    else:
        schematic_parts = schematic_parts.get("part", []) or []
        if isinstance(schematic_parts, dict):
            schematic_parts = [schematic_parts]


    check_busses = (
        xml_dict.get('eagle', {})
                .get('drawing', {})
                .get('schematic', {})
                .get('sheets', {})
                .get('sheet', {})
                .get('busses', {})
    )


    schematic_bus = check_busses.get('bus', []) if check_busses else []
    if not isinstance(schematic_bus, list):
        schematic_bus = [schematic_bus]
        

    schematic_instance = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get("sheet", {}).get('instances', {})
    

    if schematic_instance is None:
        schematic_instance = []
    else:
        schematic_instance = schematic_instance.get("instance", []) or []
        if isinstance(schematic_instance, dict):
            schematic_instance = [schematic_instance]
    
    schematic_net = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get("sheet", {}).get('nets', {})
    if schematic_net is None:
        schematic_net = []
    else:
        schematic_net = schematic_net.get("net", []) or []
        if isinstance(schematic_net, dict):
            schematic_net = [schematic_net]

    schematic_sheets = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get('sheet', {}).get('plain', {})
    schematic_setting = xml_dict.get('eagle', {}).get('drawing', {}).get('grid', {})

    # print("schematic_library:", schematic_library)
    # print("schematic_parts:", schematic_parts)
    # print("schematic_instance:", schematic_instance)
    # print("schematic_net:", schematic_net)



    schematic_info = {}
    schematic_info['IC_library'] = schematic_library
    schematic_info['parts'] = map_part_to_deviceset(schematic_parts, schematic_library, schematic_instance)
    schematic_info['nets'] = retrieve_net_info(schematic_net) if schematic_net else {}
    schematic_info['SheetInfo'] = schematic_sheets if schematic_sheets else {}
    schematic_info['setting'] = schematic_setting
    schematic_info['bus'] = schematic_bus
    # schematic_info['bus'] = schematic_bus
    # for net_name, segments in schematic_info['nets'].items():
    #     print("Net name:", net_name)
    #     print("Segments:")
    #     for segment in segments:
    #         print(segment)

    with open('example_sch.json', 'w') as f:
            json.dump(schematic_info, f, indent=2)
    
    return schematic_info



In [ ]:
def get_bounding_box_for_symbol_instance2(symbol_element, instance_x, instance_y, rot="R0", part_name="Unknown", part_value="Unknown",symbol_name="Unknown",space=10,ax=None):
    
    """
    Calculates the bounding box (xlim, ylim) for a single symbol instance.
    (No changes in this function)
    """
    # print("get_bounding_box_for_symbol_instance2")

        
    min_x_symbol = float('inf')
    max_x_symbol = float('-inf')
    min_y_symbol = float('inf')
    max_y_symbol = float('-inf')


    nested_min_x_symbol = float('inf')
    nested_max_x_symbol = float('-inf')
    nested_min_y_symbol = float('inf')
    nested_max_y_symbol = float('-inf')

    rot_angle_deg = 0.0
    if rot.upper().startswith('R180'):
        rot_angle_deg = 0
    elif rot.upper().startswith('R270'):
        rot_angle_deg = 90
    elif rot.upper().startswith('R'):
        rot_angle_deg = float(rot[1:])
    elif rot.upper().startswith('MR'):
        rot_angle_deg = 360 - float(rot[2:])
    rot_angle_rad = math.radians(rot_angle_deg)
        

    elements_list = []
    for key in ['pin', 'wire', 'rectangle', 'circle', 'polygon', 'text', 'symbol_text', 'frame']:
        for element in symbol_element.get(key, []):
            element_with_type = element.copy()
            element_with_type['element_type'] = key
            elements_list.append([element_with_type])

    # return elements_list
    bounding_boxes = []
    nest_bounding_boxes = []

    # import json
    # with open("/Users/linkaiyuan/文件/github/IMG2SCH/img2sch/Data Label/debug_symbol_element.json", "w") as f:
    #         json.dump(symbol_element, f, indent=2)
    # print("symbol_element:", json.dumps(symbol_element, indent=2))
    # print("elements_list:", elements_list)
    for elements in elements_list:
        # print("elements:", elements)
        for element in elements:
            # print("element:", element)
            element_type = element.get('element_type', 'unknown')

            # print(element_type," element:", element)
            element_min_x = float('inf')
            element_max_x = float('-inf')
            element_min_y = float('inf')
            element_max_y = float('-inf')
            
            if element_type == 'pin':
                element_min_x, element_max_x, element_min_y, element_max_y = get_pin_bounding_box(element, instance_x, instance_y, rot)
                box = {
                    'type': 'pin',
                    'x_center': (element_min_x + element_max_x) / 2,
                    'y_center': (element_min_y + element_max_y) / 2,
                    'length': element_max_x - element_min_x,
                    'width': element_max_y - element_min_y,
                    'name': element.get('name', ''),
                    'layer': element.get('layer', ''),
                }
                nest_bounding_boxes.append(box)

            elif element_type == 'wire':
                element_min_x, element_max_x, element_min_y, element_max_y = get_wire_bounding_box(element, instance_x, instance_y, rot)
                box = {
                    'type': 'wire',
                    'x_center': (element_min_x + element_max_x) / 2,
                    'y_center': (element_min_y + element_max_y) / 2,
                    'length': element_max_x - element_min_x,
                    'width': element_max_y - element_min_y,
                    'layer': element.get('layer', ''),
                }
                nest_bounding_boxes.append(box)
            

            elif element_type == 'rectangle':
                element_min_x, element_max_x, element_min_y, element_max_y = get_rectangle_bounding_box(element, instance_x, instance_y, rot)
                box = {
                    'type': 'rectangle',
                    'x_center': (element_min_x + element_max_x) / 2,
                    'y_center': (element_min_y + element_max_y) / 2,
                    'length': element_max_x - element_min_x,
                    'width': element_max_y - element_min_y,
                    'layer': element.get('layer', ''),
                }
                nest_bounding_boxes.append(box)


            elif element_type == 'circle':
                element_min_x, element_max_x, element_min_y, element_max_y = get_circle_bounding_box(element, instance_x, instance_y, rot)
                box = {
                    'type': 'circle',
                    'x_center': (element_min_x + element_max_x) / 2,
                    'y_center': (element_min_y + element_max_y) / 2,
                    'length': element_max_x - element_min_x,
                    'width': element_max_y - element_min_y,
                    'layer': element.get('layer', ''),
                }
                nest_bounding_boxes.append(box)


            elif element_type == 'polygon':
                element_min_x, element_max_x, element_min_y, element_max_y = get_polygon_bounding_box(element, instance_x, instance_y, rot)
                box = {
                    'type': 'polygon',
                    'x_center': (element_min_x + element_max_x) / 2,
                    'y_center': (element_min_y + element_max_y) / 2,
                    'length': element_max_x - element_min_x,
                    'width': element_max_y - element_min_y,
                    'layer': element.get('layer', ''),
                }
                nest_bounding_boxes.append(box)

            
            elif (element_type == 'text' and element.get('display', 'on').lower() != 'off'):
                element_min_x, element_max_x, element_min_y, element_max_y = get_attribute_text_bounding_box(element,part_name, part_value, symbol_name,ax)
                box = {
                    'type': 'text',
                    'x_center': (element_min_x + element_max_x) / 2,
                    'y_center': (element_min_y + element_max_y) / 2,
                    'length': element_max_x - element_min_x,
                    'width': element_max_y - element_min_y,
                    'name': element.get('name', ''),
                    'layer': element.get('layer', ''),
                }
                if element.get('name', '') == 'NAME' or element.get('name', '') == 'VALUE':
                    bounding_boxes.append(box)
                else:
                    nest_bounding_boxes.append(box)


            elif (element_type == 'symbol_text' and element.get('display', 'on').lower() != 'off' and not element.get('text', '').upper().startswith('>')):
                # print("symbol_text: ",element)
                element_min_x, element_max_x, element_min_y, element_max_y = get_symbol_text_bounding_box(element, instance_x, instance_y, rot)
                box = {
                    'type': 'symbol_text',
                    'x_center': (element_min_x + element_max_x) / 2,
                    'y_center': (element_min_y + element_max_y) / 2,
                    'length': element_max_x - element_min_x,
                    'width': element_max_y - element_min_y,
                    'text': element.get('text', ''),
                    'layer': element.get('layer', ''),
                }
                
                if element.get('layer', '').lower() != '94':
                    bounding_boxes.append(box)
                else:
                    nest_bounding_boxes.append(box)
            elif (element_type == 'frame'):
                # print("frame: ",element)
                element_min_x, element_max_x, element_min_y, element_max_y = get_rectangle_bounding_box(element, instance_x, instance_y, rot)
                box = {
                    'type': 'frame',
                    'x_center': (element_min_x + element_max_x) / 2,
                    'y_center': (element_min_y + element_max_y) / 2,
                    'length': element_max_x - element_min_x,
                    'width': element_max_y - element_min_y,
                    'layer': element.get('layer', ''),
                }
                nest_bounding_boxes.append(box)


        

            min_x_symbol = min(min_x_symbol, element_min_x)
            max_x_symbol = max(max_x_symbol, element_max_x)
            min_y_symbol = min(min_y_symbol, element_min_y)
            max_y_symbol = max(max_y_symbol, element_max_y)


            if element_type in ['pin', 'wire', 'rectangle', 'circle', 'polygon', 'symbol_text', 'frame'] or (
                element_type == 'text' and element.get('display', 'on').lower() != 'off' and not element.get('text', ''
                ).upper().startswith('>') and (element.get('name', '') != 'NAME' or element.get('name', '') != 'VALUE')):
                nested_min_x_symbol = min(nested_min_x_symbol, element_min_x)
                nested_max_x_symbol = max(nested_max_x_symbol, element_max_x)
                nested_min_y_symbol = min(nested_min_y_symbol, element_min_y)
                nested_max_y_symbol = max(nested_max_y_symbol, element_max_y)

            # if element_type in ['circle']:
            #     # Draw bounding box for each element if ax is provided
            #     if element_min_x != float('inf') and element_max_x != float('-inf'):
            #         x_center = (element_min_x + element_max_x) / 2
            #         y_center = (element_min_y + element_max_y) / 2
            #         length = element_max_x - element_min_x
            #         width = element_max_y - element_min_y
            #         draw_one_box(x_center, y_center, length+1, width+1, ax, color="#CE5192", thickness=2)
    nest_bounding_bbox_obect = {
        'center_x': (nested_min_x_symbol + nested_max_x_symbol) /2,
        'center_y': (nested_min_y_symbol + nested_max_y_symbol) /2,
        'length': nested_max_x_symbol - nested_min_x_symbol,
        'width': nested_max_y_symbol - nested_min_y_symbol,
        'type': 'symbol_bbox',
        'nest_bbox': nest_bounding_boxes
    }
    bounding_boxes.append(nest_bounding_bbox_obect)

            

                    
       
    # print("---------------------------------")

    if min_x_symbol == float('inf'): # No elements found
        return (instance_x, instance_x + 200), (instance_y, instance_y + 200), elements_list, bounding_boxes
    else:
        xlim = (min_x_symbol-space, max_x_symbol+space)
        ylim = (min_y_symbol-space, max_y_symbol+space)



            
        return xlim, ylim, elements_list, bounding_boxes

In [ ]:
import xml.etree.ElementTree as ET

def get_symbol_element(schematic_library, library, deviceset):
    """
    Extracts detailed gate symbols from the schematic information based on the library, deviceset, and device.
    Includes deviceset_prefix in the output.

    Args:
        schematic_library (list): The schematic information containing IC libraries.
        library (str): The name of the library.
        deviceset (str): The name of the deviceset.
        device (str): The name of the device.

    Returns:
        dict: A dictionary with the deviceset_prefix and gate details in the format:
              {deviceset_prefix: prefix, gate_name: {symbol_name: str, description: str, wire: list, text: list}}
    """
    # Extract the library data based on the library name
    library_data = next((lib for lib in schematic_library if lib['name'] == library), None)

    if library_data:
        # Extract the deviceset list
        deviceset_list = library_data.get("devicesets", {}).get("deviceset", [])
        if not isinstance(deviceset_list, list):
            deviceset_list = [deviceset_list]

        # Find the specific deviceset
        deviceset_data = next((ds for ds in deviceset_list if ds['name'] == deviceset), None)
        if deviceset_data:
            # Extract the deviceset prefix
            deviceset_prefix = deviceset_data.get("prefix", "")

            # Extract gates and their corresponding detailed symbols
            gates = deviceset_data.get("gates", {}).get("gate", [])
            if not isinstance(gates, list):
                gates = [gates]

            gate_details = {}
            for gate in gates:
                gate_name = gate['name']
                symbol_name = gate['symbol']

                # Find the detailed symbol in the library
                symbols = library_data.get("symbols", {}).get("symbol", [])
                if not isinstance(symbols, list):
                    symbols = [symbols]

                detailed_symbol = next((sym for sym in symbols if sym['name'] == symbol_name), None)
                if detailed_symbol:
                    gate_details[gate_name] = {
                        "symbol_name": symbol_name,
                        "description": detailed_symbol.get("description", ""),
                        "wire": detailed_symbol.get("wire", []),
                        "text": detailed_symbol.get("text", [])
                    }

            return {
                "deviceset_prefix": deviceset_prefix,
                **gate_details
            }
        else:
            print(f"Deviceset '{deviceset}' not found in library '{library}'.")
    else:
        print(f"Library '{library}' not found.")
    return {}


def get_text_from_symbol(schematic_library, library, deviceset_name, gate_name):
    """
    Extracts the x and y coordinates of all text elements from a symbol in the schematic library.

    Args:
        schematic_library (list): The list of library dictionaries.
        deviceset_name (str): The name of the deviceset to search for.
        device_name (str): The name of the device inside the deviceset.
        library_name (str): The name of the library to match.

    Returns:
        dict: A dictionary where keys are text strings and values are tuples (x, y),
              or an empty dictionary if not found.
    """
    symbol_element = get_symbol_element(schematic_library, library, deviceset_name).get(gate_name, {})
    # print("text_elements:",gate_name,":",symbol_element['text'])
    if symbol_element:
        text_elements = symbol_element.get('text', [])
        result = {}
        for text in text_elements:
            if isinstance(text, dict) and 'text' in text and 'x' in text and 'y' in text:
                result[text['text']] = (float(text['x']), float(text['y']))
            else:
                print(f"Invalid text element format: {text}")
        return result
    return {}


def get_position_for_new_symbol(schematic_info, new_symbol_element, spacing=20, max_width=500):
    """
    Places a new symbol using a bottom-left strategy.
    It wraps to a new column if width exceeds max_width.

    Args:
        schematic_info (dict): Schematic info including parts and IC_library.
        new_symbol_element (dict): Symbol to place.
        spacing (int): Minimum spacing between symbols.
        max_width (int): Maximum horizontal extent allowed.

    Returns:
        list: [x, y] position to place the new symbol (bottom-left placement).
    """
    parts_info = schematic_info.get("parts", {})
    ic_library = schematic_info.get("IC_library", {})

    placed_boxes = []

    min_x = float('inf')

    for part_name, part_data in parts_info.items():
        for instance in part_data.get("instances", []):
            gate = instance.get("gate", "")
            symbol = get_symbol_element_of_instance_from_library(ic_library, part_name, gate, parts_info)
            if not symbol:
                continue

            inst_x = float(instance["x"])
            inst_y = float(instance["y"])
            rot = instance.get("rot", "R0")

            xlim, ylim = get_bounding_box_for_symbol_instance(symbol, inst_x, inst_y, rot)
            placed_boxes.append((xlim, ylim))
            min_x = min(min_x, xlim[0])

    # Default starting point if no symbols placed yet
    if not placed_boxes:
        min_x = 0

    # New symbol dimensions at origin
    symbol_xlim, symbol_ylim = get_bounding_box_for_symbol_instance(new_symbol_element, 0, 0, "R0")
    symbol_width = symbol_xlim[1] - symbol_xlim[0]
    symbol_height = symbol_ylim[1] - symbol_ylim[0]

    # Bottom-left placement within width constraint
    for y in range(0, 10000):  # large enough to cover practical height
        for x in range(int(min_x), int(min_x + max_width - symbol_width) + 1):
            proposed_xlim = [x, x + symbol_width]
            proposed_ylim = [y, y + symbol_height]

            collides = False
            for existing_xlim, existing_ylim in placed_boxes:
                if not (proposed_xlim[1] + spacing <= existing_xlim[0] or
                        proposed_xlim[0] >= existing_xlim[1] + spacing or
                        proposed_ylim[1] + spacing <= existing_ylim[0] or
                        proposed_ylim[0] >= existing_ylim[1] + spacing):
                    collides = True
                    break

            if not collides:
                return [x - symbol_xlim[0], y - symbol_ylim[0]]  # Adjust for symbol origin

    return [0, 0]  # Fallback


def find_unique_part_name(schematic_file, new_symbol_element):
    """
    Finds a unique part name that does not appear in the schematic file's part list.
    The prefix is determined based on the library name.

    Args:
        schematic_file (str): Path to the schematic file.
        library_name (str): Name of the library.
        default_prefix (str): Default prefix for the part name (default is "U").

    Returns:
        str: A unique part name.
    """
    prefix = new_symbol_element.get('deviceset_prefix',"IC")
    # Process the schematic file to get the latest part list
    schematic_info = process_schematic_file(schematic_file)
    part_names = list(schematic_info['parts'].keys())

    # Find a unique part name
    index = 1
    while f"{prefix}{index}" in part_names:
        index += 1
    return f"{prefix}{index}"


def add_unit_to_schematic_full(schematic_file, library, deviceset, device, value, name, x_pos, y_pos):
    """
    Adds a new unit to the schematic file, placing the <part> element
    after the last existing part with a name like 'U[number]' within the <parts> section.

    Args:
        schematic_file (str): Path to the schematic file in XML format.
        library (str): Library name for the part.
        deviceset (str): Deviceset name.
        device (str): Device name.
        value (str): Value of the part.
        name (str): Name of the part.
        x_pos (float): X-coordinate for the instance.
        y_pos (float): Y-coordinate for the instance.
    """
    schematic_info = process_schematic_file(schematic_file)
    schematic_library = schematic_info['IC_library']
    symbol_element = get_symbol_element(schematic_library, library, deviceset)

    # Parse the schematic file
    tree = ET.parse(schematic_file)
    root = tree.getroot()

    # Navigate to the parts and instances sections
    parts_section = root.find(".//parts")
    instances_section = root.find(".//instances")

    if parts_section is None or instances_section is None:
        raise ValueError("Invalid schematic file structure. Missing parts or instances section.")

    # Check if the part already exists
    existing_part = parts_section.find(f"./part[@name='{name}']")
    if existing_part is not None:
        print(f"Part with name '{name}' already exists. Skipping addition.")
        return

    # Find the index to insert the new part
    last_u_part_index = -1
    parts_list = list(parts_section)
    for index, part in enumerate(parts_list):
        if part.tag == 'part' and part.get('name', '').startswith('U') and part.get('name', '')[1:].isdigit():
            last_u_part_index = index

    # Create the new <part> element
    part_element = ET.Element("part", {
        "name": name,
        "library": library,
        "deviceset": deviceset,
        "device": device,
        "value": value
    })
    part_element.tail = "\n"

    if last_u_part_index != -1:
        parts_section.insert(last_u_part_index + 1, part_element)
    else:
        parts_section.insert(0, part_element)

    # Ensure all parts have newlines between them
    for part in parts_section.findall("part"):
        part.tail = "\n"

    # Create attributes and instances for each gate
    for gate_name, gate_data in symbol_element.items():
        if gate_name == "deviceset_prefix":
            continue  # Skip the deviceset_prefix key

        # Create the <instance> element
        instance_element = ET.Element("instance", {
            "part": name,
            "gate": gate_name,
            "x": str(x_pos),
            "y": str(y_pos),
            "smashed": "yes"
        })
        instance_element.text = "\n"   # Add newline after opening <instance>
        instance_element.tail = "\n"   # Add newline after closing </instance>

        # Get text positions for the current gate
        text_pos = get_text_from_symbol(schematic_library, library, deviceset, gate_name)
        for text_key, text_coordinates in text_pos.items():
            if ">" in text_key:
                value_attribute_pos = text_coordinates
                sanitized_text_key = text_key.replace(">", "")

                # Add attributes for each text position
                attribute_element = ET.Element("attribute", {
                    "name": sanitized_text_key,
                    "x": str(int(x_pos - value_attribute_pos[0])),
                    "y": str(int(y_pos - value_attribute_pos[1])),
                    "size": "1.778",
                    "layer": "96",
                    "font": "vector",
                    "align": "top-left"
                })
                attribute_element.tail = "\n"
                instance_element.append(attribute_element)

        # Add instance to <instances>
        instances_section.append(instance_element)

    # Ensure all existing instances and attributes have newlines
    for inst in instances_section.findall("instance"):
        inst.tail = "\n"
        if inst.text is None or not inst.text.strip():
            inst.text = "\n"
        for child in inst:
            child.tail = "\n"

    # Write to file (compact with line-by-line structure)
    tree.write(schematic_file, encoding="utf-8", xml_declaration=True)


def add_unit_to_schematic(schematic_file,library, deviceset, device, value):
    """
    Adds a new unit to the schematic file, placing the <part> element
    after the last existing part with a name like 'U[number]' within the <parts> section.

    Args:
        schematic_file (str): Path to the schematic file in XML format.
        library (str): Library name for the part.
        deviceset (str): Deviceset name.
        device (str): Device name.
        value (str): Value of the part.
    """
    # Determine a unique part name
    schematic_info = process_schematic_file(schematic_file)
    schematic_library = schematic_info['IC_library']

    # Check if the library exists
    library_data = next((lib for lib in schematic_library if lib['name'] == library), None)
    print(library)
    if not library_data:
        print(f"Library '{library}' not found in the schematic library.")
        return

    # Check if the deviceset exists in the library
    deviceset_list = library_data.get("devicesets", {}).get("deviceset", [])
    if not isinstance(deviceset_list, list):
        deviceset_list = [deviceset_list]
    deviceset_data = next((ds for ds in deviceset_list if ds['name'] == deviceset), None)
    if not deviceset_data:
        print(f"Deviceset '{deviceset}' not found in library '{library}'.")
        return

    # Check if the device exists in the deviceset
    device_list = deviceset_data.get("devices", {}).get("device", [])
    if not isinstance(device_list, list):
        device_list = [device_list]
    device_data = next((dev for dev in device_list if dev['name'] == device), None)
    if not device_data:
        print(f"Device '{device}' not found in deviceset '{deviceset}'.")
        return
    
    new_symbol_element = get_symbol_element(schematic_info['IC_library'], library, deviceset)
    part_name = find_unique_part_name(schematic_file, new_symbol_element)

    # Get the position for the new instance
    position = get_position_for_new_symbol(schematic_info, new_symbol_element)
    print(f"Suggested position for new symbol '{part_name}': {position}")
    x_pos = position[0] # Replace with desired x-coordinate
    y_pos = position[1] # Replace with desired y-coordinate

    # Add the unit to the schematic
    add_unit_to_schematic_full(schematic_file, library, deviceset, device, value, part_name, x_pos, y_pos)

In [ ]:
import os
import csv
import traceback

def _append_csv_row(csv_path, fieldnames, row):
    write_header = not os.path.exists(csv_path) or os.path.getsize(csv_path) == 0
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            w.writeheader()
        w.writerow(row)
        f.flush() 
def get_lib_inst_from_schematic(sch_file, log_csv):
    """
    Extracts parts from a schematic file and returns their bounding boxes.
    """
    if not os.path.exists(sch_file):
        raise FileNotFoundError(f"Schematic file not found: {sch_file}")

    records = []
    fieldnames = [
    'sch_file', 'library', 'symbol', 'device', 'deviceset', 'package', 'gates', 'tech']
    list_of_schematic_info = process_schematic_file(sch_file)
    for i, schematic_info in enumerate(list_of_schematic_info):
        # import json
        # with open("/Users/linkaiyuan/文件/github/IMG2SCH/img2sch/Data Label/checkschematic_info.json", "w") as f:
        #             json.dump(schematic_info['parts'], f, indent=4, default=_json_default)

        # render and save
        # sizes    = visualize_schematic(schematic_info)
        # sizes   = visualize_schematic_nobounding(sch_file, save_path=png_path)
        # px_per_x, px_per_y, _, _, img_w, img_h,_,_,_ = sizes


        ic_library = schematic_info.get('IC_library')
        parts_info = schematic_info.get('parts')


        for lib in ic_library:
            lib_name = lib.get("name", "")
            devicesets = lib.get("devicesets", {}).get("deviceset", {})
            if isinstance(devicesets, dict):
                devicesets = [devicesets]
            if isinstance(devicesets, list):
                for ds in devicesets:
                        
                        gates = ds.get("gates", {})
                        deviceset_name = ds.get("name", "")
                        devices = ds.get("devices", {}).get("device")
                        if isinstance(devices, dict):
                            devices = [devices]
                            
                        for device in devices:
                            device_name = device.get("name", "")
                            packages = device.get("package", "")
                            base_name    = os.path.splitext(os.path.basename(sch_file))[0]
                            
                            # print(gates)
                            technology = device.get("technologies", [])
                            if isinstance(technology, dict):
                                technology = [technology]
                            all_techs = []
                            for tech in technology:
                                tech_name = tech.get("name", "")
                                if not tech_name:
                                    tech_name = "Unknown"
                                tech_value = tech.get("value", "")
                                if not tech_value:
                                    tech_value = "Unknown"
                                all_techs.append({
                                    "tech_name": tech_name,
                                    "tech_value": tech_value
                                })

                            if gates is None:
                                print("wrong device:", device_name)
                                row = {
                                    'sch_file': base_name,
                                    'library': lib_name,
                                    'symbol': "",
                                    'device': device_name,
                                    'deviceset': deviceset_name,
                                    'package': packages,
                                    'tech': all_techs,
                                    'gates': "",
                                }
                                _append_csv_row(log_csv, fieldnames, row)
                                records.append(row)
                                continue
                            
                            all_gates = gates.get("gate", {})
                            if isinstance(all_gates, dict):
                                all_gates = [all_gates]
                            
                            for gate in all_gates:
                                gate_name = gate.get("name", "")
                                symbol = gate.get("symbol", "")
                                if isinstance(symbol, list):
                                    symbol = symbol[0]
                                row = {
                                    'sch_file': base_name,
                                    'library': lib_name,
                                    'symbol': symbol,
                                    'device': device_name,
                                    'deviceset': deviceset_name,
                                    'package': packages,
                                    'tech': all_techs,
                                    'gates': gate_name,
                                }
                    
                                _append_csv_row(log_csv, fieldnames, row)
                                records.append(row)
    
    return records


def save_all_deivces_to_csv(sch_folder, log_csv):
    START_INDEX = 0
    fieldnames = [
    'sch_file', 'part_name', 'library', 'symbol', 'x_inst', 'y_inst', 'rot',
    'norm_x_center', 'norm_y_center', 'norm_width', 'norm_height',
    'part_value', 'device', 'deviceset', 'package'
]
    sch_files = sorted([f for f in os.listdir(sch_folder) if f.lower().endswith(".sch")])
    sch_files = sch_files[START_INDEX:]
    records = []
    for idx, sch_file in enumerate(sch_files):
        sch_file = os.path.join(sch_folder, sch_file)
        try:
            row = get_lib_inst_from_schematic(sch_file, log_csv)
        except Exception as e:
            row["status"] = "error"
            row["error"] = str(e)
            row["traceback"] = "".join(traceback.format_exc()[-2000:])
            print(f"  → Failed: {e}")

        
    print(f"Per-file log written continuously to: {log_csv}")
